# prototype3 — if1 inside Blocks 1–5 (BOOTSTRAP_V9)

Opening a notebook from GitHub **does not** include `src/`. **Runtime → Run all.** The first code cell must print `BOOTSTRAP_V9` and `if1` in `run_blocks_1_to_5`. If you still see `No module named med_doc`, open this file from GitHub branch **`block1`**, then **Runtime → Disconnect and delete runtime**.

**Default `PIPELINE = "if1"`** runs the **working** `run_blocks_1_to_5(..., if1=True)` path: if1block1 bar+gutter warp and if1block4 coverage high/low are **embedded** in Blocks 1–5. Block 5 still writes `order.json` (`ordered_tests` = high-confidence ticks). Set `PIPELINE = "live"` for unchanged Blocks 1–5 (`if1=False`).

**Do not upload clinic PHI to Colab.**


Per-block notebooks: [1](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_1_Document_Normalization.ipynb) · [2](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_2_Knowledge_Graph.ipynb) · [3](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_3_Marks_and_HTR.ipynb) · [4](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_4_KG_Rescoring.ipynb) · [5](https://colab.research.google.com/github/RwaRwa599/epq3/blob/block1/notebooks/Block_5_Review_and_LIS.ipynb)


## 0. Download src/med_doc (zipball)


In [ ]:
# BOOTSTRAP_V9 — zipball of branch block1 (if1 embedded in Blocks 1–5)
import importlib
import inspect
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

CONTENT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = CONTENT / "epq3"
SRC = REPO / "src"
URL = "https://codeload.github.com/RwaRwa599/epq3/zip/refs/heads/block1"

zpath = CONTENT / "epq3-block1.zip"
print("Downloading", URL)
urllib.request.urlretrieve(URL, zpath)
extract = CONTENT / "_epq3_extract"
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(extract)
found = list(extract.glob("*/src/med_doc/__init__.py"))
if not found:
    raise RuntimeError(f"zip missing src/med_doc: {list(extract.iterdir())}")
unpacked = found[0].parents[2]
if REPO.exists():
    shutil.rmtree(REPO)
shutil.move(str(unpacked), str(REPO))
shutil.rmtree(extract, ignore_errors=True)
zpath.unlink(missing_ok=True)

src = str(SRC.resolve())
while src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
os.chdir(REPO)
for name in list(sys.modules):
    if name == "med_doc" or name.startswith("med_doc."):
        del sys.modules[name]
importlib.invalidate_caches()
import med_doc
from med_doc.pipeline import run_blocks_1_to_5
from med_doc.if1 import run_if1

print("BOOTSTRAP_V9")
print("cwd:", os.getcwd())
print("med_doc:", med_doc.__file__)
print("live_params:", list(inspect.signature(run_blocks_1_to_5).parameters))
print("if1 embedded:", "if1" in inspect.signature(run_blocks_1_to_5).parameters)
print("run_if1 wraps live:", inspect.signature(run_if1))
from med_doc.htr.marks import TICK_POLICY
print("tick_policy:", TICK_POLICY)
if TICK_POLICY != "slash-v2":
    raise RuntimeError(
        f"stale med_doc tick_policy={TICK_POLICY!r}. Disconnect and delete runtime, "
        "re-open prototype3.ipynb from GitHub branch block1."
    )
from med_doc.if1 import load_groups
print("if1 groups:", len(load_groups()))


In [ ]:
# Runtime deps via pip CLI (not %pip / not pip -e — those restart Colab mid-run).
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pydantic", "matplotlib", "Pillow", "numpy"]
)


The repo is public. No GitHub token is required. Opening this notebook from GitHub still does not include `src/` — the first cell downloads the zipball.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V9 "
            "and med_doc: .../src/med_doc/__init__.py. Open prototype3.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet


## 1. Choose a batch of images

Same input shapes as Block 1a: a **folder**, a **ZIP of photos**, a **list of paths**, or Colab multi-upload.

`PIPELINE = "if1"` (default) calls **`run_blocks_1_to_5(..., if1=True)`** so if1 warp + coverage tiers produce Block 5 `order.json`. `"live"` is `if1=False`.

Do **not** upload clinic PHI. Default: two copies of the synthetic blank.


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V9 "
            "and med_doc: .../src/med_doc/__init__.py. Open prototype3.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

import shutil
from med_doc.pipeline import run_blocks_1_to_5

USE_UPLOAD = False
PIPELINE = "if1"  # "if1" experimental; "live" = original Blocks 1–5

OUTPUT_MODE = "dev"  # overlays in ZIPs; "user" = order.json only
BATCH_DIR = None  # e.g. Path("/content/photos") or Path("photos.zip")

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    names = list(uploaded.keys())
    if len(names) == 1 and names[0].lower().endswith(".zip"):
        batch_input = names[0]
    else:
        batch_input = names
elif BATCH_DIR is not None:
    batch_input = BATCH_DIR
else:
    sheet = demo_sheet()
    raw = OUT / "raw_batch"
    raw.mkdir(parents=True, exist_ok=True)
    shutil.copy2(sheet, raw / "form_a.png")
    shutil.copy2(sheet, raw / "form_b.png")
    batch_input = raw

print("batch_input:", batch_input)


## 2. Run the working pipeline (if1 embedded or live)

One runner: `run_blocks_1_to_5`. With `if1=True` it uses if1block1/3/4 **inside** Blocks 1–5, then original Block 5 (`order.json`, high-confidence ticks only).


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V9 "
            "and med_doc: .../src/med_doc/__init__.py. Open prototype3.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from med_doc.pipeline import run_blocks_1_to_5

RUN_DIR = OUT / PIPELINE
RUN_DIR.mkdir(parents=True, exist_ok=True)

pipe = run_blocks_1_to_5(
    batch_input,
    output_dir=RUN_DIR,
    backend="lexicon",
    patch_missing_edta=False,
    output_mode=OUTPUT_MODE,
    if1=(PIPELINE == "if1"),
    htr_mode="nonverbal" if PIPELINE == "if1" else "both",
    mark_backend="geometry",
    vision_backend="off",
    combo_backend="off",
)
print("if1", pipe.get("if1"), "mode", pipe["output_mode"], "docs", pipe["block5"]["manifest"]["total_documents"])
print("output_json", pipe.get("output_json"))
b1_docs = pipe["block1"]["manifest"]["documents"]
for row in b1_docs:
    print(row["doc_id"], "warp", row.get("warp_method"), "align", row.get("alignment_confidence"), "block", pipe["block1"]["manifest"].get("block"))
for row in pipe["block5"]["manifest"]["documents"]:
    print(row["doc_id"], "ticked", row.get("n_ticked"), "needs_review", row.get("needs_review"),
          "observed", row.get("observed_tubes"))


## 3. Inspect overlays and (if1) high/low order lists


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V9 "
            "and med_doc: .../src/med_doc/__init__.py. Open prototype3.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()


import json

root = Path(pipe["output_dir"])
bundle = json.loads(Path(pipe["output_json"]).read_text())
print("orders", bundle["total_documents"], "file", pipe["output_json"])
for order in bundle["orders"]:
    print(
        order["doc_id"],
        "needs_review", order["needs_review"],
        "reg_fail", order.get("registration_failure_suspected"),
        "conf", order.get("overall_confidence"),
        "retry", order.get("crop_retry_rate"),
        "n_ticked", len(order.get("ticked_test_ids") or []),
        "n_ordered", len(order.get("ordered_tests") or []),
        "high", order.get("ordered_tests_high"),
        "low", order.get("ordered_tests_low"),
        "initial", order.get("initial_ticked_test_ids"),
        "observed", order["observed_tubes"],
        "reasons", order.get("review_reasons"),
    )

if pipe["output_mode"] != "dev":
    print("set OUTPUT_MODE=dev for overlays")
else:
    docs = pipe["block5"]["manifest"]["documents"]
    show = docs[:3]
    for row in show:
        doc_id = row["doc_id"]
        overlay = root / "b1" / "docs" / doc_id / "overlay.png"
        annotated = root / "b3" / "docs" / doc_id / "annotated_canvas.png"
        pred_path = root / "b4" / "docs" / doc_id / "prediction.json"
        hyp_path = root / "b3" / "docs" / doc_id / "hypotheses.json"
        meta_path = root / "b1" / "docs" / doc_id / "metadata.json"
        print("debug", doc_id, "overlay", overlay.exists(), "annotated", annotated.exists())
        if overlay.exists():
            show_rgb(overlay, f"overlay — {doc_id}", figsize=(12, 10))
        if annotated.exists():
            show_rgb(annotated, f"annotated — {doc_id}", figsize=(12, 10))
        if meta_path.exists():
            meta = json.loads(meta_path.read_text())
            warp = (meta.get("warp") or meta.get("extra", {}).get("warp") or {})
            print(doc_id, "warp_method", meta.get("warp_method") or warp.get("method"))
        if hyp_path.exists():
            hyp = json.loads(hyp_path.read_text())
            print(doc_id, "initial_ticked", hyp.get("initial_ticked_test_ids"))
            print(doc_id, "marked", [k for k, v in (hyp.get("nonverbal") or {}).items() if v.get("is_marked")])
        if pred_path.exists():
            pred = json.loads(pred_path.read_text())
            print(doc_id, "high", pred.get("ordered_tests_high"))
            print(doc_id, "low", pred.get("ordered_tests_low"))
            print(doc_id, "LIS ticked", pred.get("ticked_test_ids"))
            combo = pred.get("combo_scores") or {}
            print(doc_id, "combo m", combo.get("m"))
            print(doc_id, "combo e", combo.get("e"))


## 4. Download — `order.json` (user) or ZIPs (dev)


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V9 "
            "and med_doc: .../src/med_doc/__init__.py. Open prototype3.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()


root = Path(pipe["output_dir"])
if pipe["output_mode"] == "dev":
    names = ["block1.zip", "block3.zip", "block4.zip", "block5.zip"]
    if PIPELINE == "if1":
        names += ["if1block1.zip", "if1block3.zip", "if1block4.zip"]
    for name in names:
        download(root / name)
else:
    download(root / "order.json")
